In [2]:
# !pip install pypdf langchain-community langchain-huggingface sentence-transformers faiss-cpu

- PyPDFLoader ----> Open PDF and read its text page by page.
- RecursiveCharacterTextSplitter ----> cut's the text into smaller chunks intelligently.
- HuggingFaceEmbeddings ----> Converts each chunk into a vector
- FAISS ----> I will search all those backlots so we can search them later. 

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

### Loading pdf

In [10]:
def load_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path) # loads pdf
    documents = loader.load()
    print(f"Loaded {len(documents)} pages")
    return documents



docs = load_pdf('hand on llms modified.pdf') # pdf path
print(docs[0].page_content[:500])

Loaded 105 pages
CHAPTER 3
Looking Inside Large Language Models
Now that we have a sense of tokenization and embeddings, we’re ready to dive deeper
into the language model and see how it works. In this chapter, we’ll look at some of
the main intuitions of how Transformer language models work. Our focus will be on
text generation models so we get a deeper sense for generative LLMs in particular.
We’ll be looking at both the concepts and some code examples that demonstrate
them. Let’s start by loading a language m


### Chuncking - **Now we turn pages into chunks**

In [22]:
def chunk_documents(documents, chunk_size=800, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
    )

    chunks = text_splitter.split_documents(documents)
    print(f"Created {len(chunks)} chunks")
    return chunks


chunks = chunk_documents(docs)
print(chunks[0].page_content[:500])
print(chunks[0].metadata)

Created 261 chunks
CHAPTER 3
Looking Inside Large Language Models
Now that we have a sense of tokenization and embeddings, we’re ready to dive deeper
into the language model and see how it works. In this chapter, we’ll look at some of
the main intuitions of how Transformer language models work. Our focus will be on
text generation models so we get a deeper sense for generative LLMs in particular.
We’ll be looking at both the concepts and some code examples that demonstrate
them. Let’s start by loading a language m
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260607201226', 'source': 'hand on llms modified.pdf', 'total_pages': 105, 'page': 0, 'page_label': '1'}


### Add Embeddings and build FAISS index

In [24]:
def get_embeddings_model():
    embeddings = HuggingFaceEmbeddings(
        model_name= 'sentence-transformers/all-MiniLM-L6-v2'
    )
    return embeddings

def build_faiss(chunks, embeddings):
    vector_store = FAISS.from_documents(chunks, embeddings)
    return vector_store

def save_faiss(vector_store, folder_path = 'faiss_index'):
    vector_store.save_local(folder_path)
    print(f"Saved FAISS index to {folder_path}")


embeddings = get_embeddings_model()
vector_store = build_faiss(chunks, embeddings)
save_faiss(vector_store, folder_path='faiss_llm_book')




modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Saved FAISS index to faiss_llm_book


### Load FAISS index

In [29]:
def load_faiss(folder_path='faiss_llm_book'):
    embeddings = get_embeddings_model()
    vector_store = FAISS.load_local(
        folder_path,
        embeddings,
        allow_dangerous_deserialization=True,
    )
    return vector_store

db = load_faiss()
query = 'what is the main usage of RAG?'
results = db.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"Result{i} : {doc.page_content[:400]}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Result1 : Figure 8-24. A basic RAG pipeline is made up of a search step followed by a grounded
generation step where the LLM is prompted with the question and the information
retrieved from the search step.
RAG systems incorporate search capabilities in addition to generation capabilities.
They can be seen as an improvement to generation systems because they reduce
their hallucinations and improve their fac
Result2 : Advanced RAG Techniques
There are several additional techniques to improve the performance of RAG systems.
Some of them are laid out here.
Query rewriting
If the RAG system is a chatbot, the preceding simple RAG implementation would
likely struggle with the search step if a question is too verbose, or to refer to context
in previous messages in the conversation. This is why it’s a good idea to use
Result3 : eration model on a specific dataset.
Figure 8-3. A RAG system formulates an answer to a question and (preferably) cites its
information sources.
The rest of the chapter

### Using Generation model via API(Google-Genai) 

In [ ]:
#! pip install -U "langchain[google-genai]"

   ---------------------------------------- 0.0/832.5 kB ? eta -:--:--
   ------------------------------------- -- 786.4/832.5 kB 5.5 MB/s eta 0:00:01
   ---------------------------------------- 832.5/832.5 kB 3.9 MB/s  0:00:00

   ---------- ----------------------------- 1/4 [google-auth]
   ---------- ----------------------------- 1/4 [google-auth]
   -------------------- ------------------- 2/4 [google-genai]
   -------------------- ------------------- 2/4 [google-genai]
   -------------------- ------------------- 2/4 [google-genai]
   -------------------- ------------------- 2/4 [google-genai]
   -------------------- ------------------- 2/4 [google-genai]
   -------------------- ------------------- 2/4 [google-genai]
   -------------------- ------------------- 2/4 [google-genai]
   -------------------- ------------------- 2/4 [google-genai]
   -------------------- ------------------- 2/4 [google-genai]
   -------------------- ------------------- 2/4 [google-genai]
   --------------

In [38]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ.get("GOOGLE_API_KEY")


In [40]:
def building_rag_chain():
    vectorstore = load_faiss()
    retriever = vectorstore.as_retriever(search_kwargs={"k":4})
    llm =ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

    from langchain_core.prompts import ChatPromptTemplate
    # create a prompt template 
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a helpful assistant for question answering. "
            "Answer the user's question only from the provided context. "
            "If the answer is not in the context, say you don't know. "
            "Keep the answer clear and concise.\n\n"
            "Context:\n{context}"

        ),
        ("human", "{input}")
    ])
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain
    from langchain_classic.chains import create_retrieval_chain
    document_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, document_chain)
    return rag_chain

rag_chain = building_rag_chain()
query = "what is rag and why to use it?"
response = rag_chain.invoke({"input": query})

print("\nQUESTION:\n", query)
print("\nANSWER:\n", response["answer"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


QUESTION:
 what is rag and why to use it?

ANSWER:
 RAG (Retrieval-Augmented Generation) systems incorporate search capabilities in addition to generation capabilities. They are used to reduce hallucinations and improve factuality in LLMs, and enable use cases like "chat with my data."
